# 06 - Couverture signature-Fourier sous shifted Bergomi

Sous Bergomi rough la volatilite est **non-markovienne** : on represente
`sigma_t` dans l'algebre tensorielle (signatures du brownien augmente `W_hat`)
et on resout une **equation de Riccati signature** pour obtenir prix et delta par
Fourier.

Deux familles d'approches (fideles a `stage00`) :

1. **sigma analytique rescalee** : garde la formule exacte de `sigma_t`, resout
   la Riccati dans une base rescalee (pas de regression). -> options europeenne et asiatique.
2. **sigma apprise** : `sigma_t` approchee par regression Ridge sur les
   signatures, puis Riccati. -> options europeenne **et asiatique**.

> L'option **lookback** n'a pas de formule signature-Fourier dans `stage00` : elle
> n'est donc pas traitee ici (voir le deep hedging, notebook 04, pour le lookback).

Tailles completes : M=5, depth=5, I_test=20000, J=126.
**Attention : ces calculs sont longs (plusieurs minutes par delta).**

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from fourier.signature_volatility import (
    build_scale_vector,
    compute_X0_sig_fourier,
    compute_X0_sig_fourier_asian,
    compute_X0_sig_fourier_asian_rescaled,
    compute_X0_sig_fourier_rescaled,
    compute_delta_sig_fourier_asian_rescaled,
    compute_delta_sig_fourier_asian_vectorized,
    compute_delta_sig_fourier_rescaled,
    compute_delta_sig_fourier_vectorized,
)
from models import compute_payoff_asian, compute_payoff_europeen, simulate_shifted_bergomi
from signatures import compute_signatures, generate_words, word_to_index, zeros_tensor
from utils.plotting import plot_pnl_distribution

In [2]:
# Parametres shifted Bergomi
S0, f0, K = 1.0, 0.0625, 1.0
rho, varsigma, epsilon, H = -0.7, 1.6, 1/52, 0.1
V0 = f0
T, J = 1/12, 126
I_train, I_test = 10_000, 20_000

# Algebre tensorielle (ordre de troncature M=5)
M = 5
words = generate_words(M)
word_idx = word_to_index(words)
words_no_empty = [w for w in words if len(w) > 0]
dt = T / J
scale = build_scale_vector(words, dt)
scale_no_empty = np.array([scale[word_idx[w]] for w in words_no_empty])
print('M =', M, '| nb mots =', len(words), '| nb mots non vides =', len(words_no_empty))

M = 5 | nb mots = 63 | nb mots non vides = 62


In [3]:
S_train, dS_train, sigma_train, states_train, W_hat_train = simulate_shifted_bergomi(
    S0, varsigma, f0, epsilon, H, rho, T, J, I_train, seed=42)
S_test, dS_test, sigma_test, states_test, W_hat_test = simulate_shifted_bergomi(
    S0, varsigma, f0, epsilon, H, rho, T, J, I_test, seed=123)

payoff_eu_test = compute_payoff_europeen(S_test, K)
payoff_as_test = compute_payoff_asian(S_test, K)
print('W_hat_test:', W_hat_test.shape)

W_hat_test: (20000, 127, 2)


In [4]:
# Signatures de W_hat (depth=M), alignees sur t_0,...,t_J (signature nulle en t_0)
sig_stream_train = compute_signatures(W_hat_train, depth=M)
sig_stream_test  = compute_signatures(W_hat_test,  depth=M)

sig_W_full_train = np.zeros((I_train, J + 1, sig_stream_train.shape[-1]))
sig_W_full_test  = np.zeros((I_test,  J + 1, sig_stream_test.shape[-1]))
sig_W_full_train[:, 1:, :] = sig_stream_train
sig_W_full_test[:, 1:, :]  = sig_stream_test

# Version rescalee : <psi, sig> = <psi_tilde, sig_tilde>
sig_W_tilde_test = sig_W_full_test / scale_no_empty
print('sig_W_full_test:', sig_W_full_test.shape)

sig_W_full_test: (20000, 127, 62)


## Approche 1 : sigma analytique rescalee


In [ ]:
f_eur = lambda t, u: 1j * u

X0_resc = compute_X0_sig_fourier_rescaled(
    rho, f_eur, T, J, M, words, word_idx, varsigma, f0, epsilon, H, scale,
    V0=V0, K=K, n_laguerre=32, N_steps=10)

delta_resc = compute_delta_sig_fourier_rescaled(
    rho, S_test, K, f_eur, sig_W_tilde_test, sigma_test, T, J, M, words, word_idx,
    varsigma, f0, epsilon, H, scale, V0=V0, N_steps=10, n_laguerre=32)

pnl_resc = X0_resc + np.sum(delta_resc * dS_test, axis=1) - payoff_eu_test
print(f'X0 rescaled = {X0_resc:.6f}   (MC = {payoff_eu_test.mean():.6f})')
print(f'European (rescaled)  MSE = {np.mean(pnl_resc**2):.3e}   cible papier ~ 8.16e-05')

### Asiatique (sigma analytique rescalee)


In [5]:
f_as = lambda t, u: 1j * u * (T - t) / T

X0_as_resc = compute_X0_sig_fourier_asian_rescaled(
    rho, f_as, T, J, M, words, word_idx, varsigma, f0, epsilon, H, scale,
    V0=V0, K=K, n_laguerre=32, N_steps=10)

delta_as_resc = compute_delta_sig_fourier_asian_rescaled(
    rho, S_test, K, f_as, sig_W_tilde_test, sigma_test, T, J, M, words, word_idx,
    varsigma, f0, epsilon, H, scale, V0=V0, N_steps=10, n_laguerre=32)

pnl_as_resc = X0_as_resc + np.sum(delta_as_resc * dS_test, axis=1) - payoff_as_test
print(f'X0 Asian rescaled = {X0_as_resc:.6f}   (MC = {payoff_as_test.mean():.6f})')
print(f'Asian (rescaled)  MSE = {np.mean(pnl_as_resc**2):.3e}')


X0 Asian rescaled = 0.014944   (MC = 0.014905)
Asian (rescaled)  MSE = 1.778e-05


## Approche 2 : sigma apprise par regression Ridge

On apprend `sigma_t ~ <sigma_learned, signature(W_hat)_t>` par regression
lineaire (Ridge) sur les trajectoires d'entrainement, puis on construit le
tenseur `sigma_learned` dans `T^M`.

In [ ]:
X = sig_W_full_train[:, :J, :].reshape(-1, sig_stream_train.shape[-1])
y = sigma_train[:, :J].reshape(-1)

reg = make_pipeline(StandardScaler(), Ridge(alpha=1e-8))
reg.fit(X, y)
ridge = reg.named_steps['ridge']
print(f'R2 regression sigma = {reg.score(X, y):.4f}')

sigma_learned = zeros_tensor(M)
sigma_learned[word_idx[()]] = ridge.intercept_
for k, w in enumerate(words_no_empty):
    sigma_learned[word_idx[w]] = ridge.coef_[k]
print(f'norme sigma_learned = {np.linalg.norm(sigma_learned):.4f}')

### Europeenne (sigma apprise)

In [ ]:
f_eur = lambda t, u: 1j * u

X0_eu = compute_X0_sig_fourier(
    rho, f_eur, sigma_learned, T, M, words, word_idx, V0=V0, K=K, n_laguerre=32, N_steps=100)

delta_eu = compute_delta_sig_fourier_vectorized(
    rho, S_test, K, f_eur, sig_W_full_test, sigma_learned, sigma_test,
    T, J, M, words, word_idx, V0=V0, N_steps=50, n_laguerre=32)

pnl_eu = X0_eu + np.sum(delta_eu * dS_test, axis=1) - payoff_eu_test
print(f'X0 learned = {X0_eu:.6f}   (MC = {payoff_eu_test.mean():.6f})')
print(f'European (learned)  MSE = {np.mean(pnl_eu**2):.3e}')

### Asiatique (sigma apprise)

In [ ]:
f_as = lambda t, u: 1j * u * (T - t) / T

X0_as = compute_X0_sig_fourier_asian(
    rho, f_as, sigma_learned, T, M, words, word_idx, V0=V0, K=K, n_laguerre=32, N_steps=100)

delta_as = compute_delta_sig_fourier_asian_vectorized(
    rho, S_test, K, f_as, sig_W_full_test, sigma_learned, sigma_test,
    T, J, M, words, word_idx, V0=V0, N_steps=50, n_laguerre=32)

pnl_as = X0_as + np.sum(delta_as * dS_test, axis=1) - payoff_as_test
print(f'X0 Asian learned = {X0_as:.6f}   (MC = {payoff_as_test.mean():.6f})')
print(f'Asian (learned)  MSE = {np.mean(pnl_as**2):.3e}')

## Distributions de P&L

In [ ]:
plot_pnl_distribution([
    (pnl_resc, 'Sig-Fourier rescaled', 'blue',  'solid'),
    (pnl_eu,   'Sig-Fourier learned',  'green', 'dashed'),
], title='European (Bergomi)')

plot_pnl_distribution([
    (pnl_as_resc, 'Sig-Fourier rescaled', 'blue', 'solid'),
    (pnl_as,      'Sig-Fourier learned',  'green', 'dashed'),
], title='Asian (Bergomi)')
